In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

base = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/titanic/"
def load(name):
    path = Path("dataset/titanic") / name
    return pd.read_csv(path if path.exists() else base + name)

train, test = load("train.csv"), load("test.csv")
torch.manual_seed(42)

In [8]:
def features(df):
    x = df[["Age", "SibSp", "Parch", "Fare", "Pclass", "Sex", "Embarked"]].copy()
    title = df["Name"].str.extract(r",\s*([^.]*)\.")[0].str.strip()
    x["Title"] = title.where(title.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")
    x["HasCabin"] = df["Cabin"].notna().astype(int)
    return x

num = ["Age", "SibSp", "Parch", "Fare", "HasCabin"]
cat = ["Pclass", "Sex", "Embarked", "Title"]

def fit(X, y, epochs=200):
    prep = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                              OneHotEncoder(handle_unknown="ignore", sparse_output=False)), cat),])

    x = torch.tensor(prep.fit_transform(X), dtype=torch.float32, device=device)
    y = torch.tensor(y.to_numpy(), dtype=torch.float32, device=device).view(-1, 1)

    model = nn.Sequential(nn.Linear(x.shape[1], 32), nn.ReLU(), nn.Linear(32, 1)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.BCEWithLogitsLoss()

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()

    return model, prep

def predict(model, prep, X):
    x = torch.tensor(prep.transform(X), dtype=torch.float32, device=device)
    model.eval()
    with torch.no_grad():
        return (model(x).sigmoid().squeeze(1) >= 0.5).cpu().numpy().astype(int)

In [9]:
X, y = features(train), train["Survived"]
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model, prep = fit(X_tr, y_tr)
accuracy = (predict(model, prep, X_val) == y_val.to_numpy()).mean()
print(f"Validation Accuracy: {accuracy:.4f}")

Validation Accuracy: 0.8156


In [10]:
model, prep = fit(X, y)
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": predict(model, prep, features(test)),
})
submission.to_csv("submission_nearal_network.csv", index=False)
print("Saved submission")
submission.head()

Saved submission


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
